# Companion: Contextualized Representations and Text Embeddings

This notebook accompanies the material on *Contextualized Representations and Text Embeddings*. Each section matches a core didactic block:

1. **Contextual Token Occurrences vs. Static Types:** Tracking word vectors across layers; how context separates polysemous senses.
2. **Masked Language Modeling (MLM):** The corruption mechanism, prediction heads, co-occurrence learning, and syntactic agreement.
3. **From Tokens to Texts: Fine-Tuned Embeddings vs. Naive Pooling:** Why raw BERT pooling scores paraphrase and topical pairs alike, and how metric fine-tuning separates them.
4. **The InfoNCE Objective and Temperature:** In-batch negative matrix math, temperature scaling $\tau$, per-row losses, and independent matrix practice.
5. **Retrieval in Practice: Bi-Encoder, Lexical Traps, and Cross-Encoder Reranking:** Exact dense vector search vs. TF-IDF lexical search and cross-encoder rescoring.
6. **Modern Horizons: Matryoshka Representation Learning & Late Interaction:** Truncating dimension prefixes with a Matryoshka-trained model, and ColBERT's token-level MaxSim operator on real contextual vectors.
7. **Shared Multimodal Embedding Spaces (EmbeddingGemma):** Cross-modal alignment across text, image, and speech in a single 768-dimensional space (e.g. banana vs. apple across modalities).

The notebook supports conceptual understanding through concrete code experiments; it is not a graded assignment.

- **Core (approx. 50–60 minutes):** Parts 1, 2, 3, and 4.
- **Explore (optional, approx. 35–45 minutes):** Parts 5, 6, and 7.

Cells marked **Your Turn** provide active self-checks with starter code. Runs on any standard Python environment or Google Colab CPU.

In [ ]:
# Install required packages if running in a fresh environment / Google Colab
# !pip install -q transformers sentence-transformers scikit-learn torch numpy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)
torch.manual_seed(42)

## 1. Contextual Token Occurrences vs. Static Types *(Core)*

In static word embeddings (such as word2vec or GloVe), each vocabulary type receives a single fixed vector $\mathbf{w} \in \mathbb{R}^d$. A polysemous word (e.g. *bank*) is forced into a static compromise vector regardless of whether it refers to a river edge or a financial institution.

In a contextual encoder (like BERT), the representation is a function of the entire sequence:
$$\mathbf{h}_i = f_{\theta}(w_1, \dots, w_n)_i$$

Let us track the occurrence representation of the token **"bank"** across three sentences:
1. River 1: *"The river bank was muddy and overflowing after heavy rain."*
2. River 2: *"We sat on the grassy bank watching boats pass along the water."*
3. Finance: *"He walked into the bank to deposit cash into his account."*

In [ ]:
from transformers import BertTokenizer, BertModel

# Load standard bert-base-uncased with all intermediate hidden states enabled
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
model.eval()

sent_river = "The river bank was muddy and overflowing after heavy rain."
sent_river2 = "We sat on the grassy bank watching boats pass along the water."
sent_money = "He walked into the bank to deposit cash into his account."

inputs_river = tokenizer(sent_river, return_tensors="pt")
inputs_river2 = tokenizer(sent_river2, return_tensors="pt")
inputs_money = tokenizer(sent_money, return_tensors="pt")

with torch.no_grad():
    out_river = model(**inputs_river)
    out_river2 = model(**inputs_river2)
    out_money = model(**inputs_money)

# Locate token position for 'bank' in each sentence
id_bank = tokenizer.convert_tokens_to_ids("bank")
idx_river = (inputs_river.input_ids[0] == id_bank).nonzero().item()
idx_river2 = (inputs_river2.input_ids[0] == id_bank).nonzero().item()
idx_money = (inputs_money.input_ids[0] == id_bank).nonzero().item()

num_layers = len(out_river.hidden_states)  # 13: Layer 0 (input embeddings) + 12 transformer layers
print(f"Total layers available (0 to 12): {num_layers}")

Total layers available (0 to 12): 13


In [ ]:
# Track cosine similarities layer by layer with widened column formatting
print(f"{'Layer':<12}{'Cosine Sim (River vs. Finance)':<35}{'Cosine Sim (River 1 vs. River 2)':<35}")
print("-" * 82)

for layer in range(num_layers):
    v_riv = out_river.hidden_states[layer][0, idx_river]
    v_riv2 = out_river2.hidden_states[layer][0, idx_river2]
    v_mon = out_money.hidden_states[layer][0, idx_money]
    
    sim_diff_sense = F.cosine_similarity(v_riv, v_mon, dim=0).item()
    sim_same_sense = F.cosine_similarity(v_riv, v_riv2, dim=0).item()
    
    layer_name = "L0 (Input)" if layer == 0 else f"Layer {layer}"
    print(f"{layer_name:<12}{sim_diff_sense:<35.4f}{sim_same_sense:<35.4f}")

Layer       Cosine Sim (River vs. Finance)     Cosine Sim (River 1 vs. River 2)   
----------------------------------------------------------------------------------
L0 (Input)  0.9546                             0.9502                             
Layer 1     0.6830                             0.8509                             
Layer 2     0.5959                             0.8429                             
Layer 3     0.5710                             0.8094                             
Layer 4     0.5203                             0.7708                             
Layer 5     0.4867                             0.7551                             
Layer 6     0.4922                             0.7261                             
Layer 7     0.4681                             0.7253                             
Layer 8     0.4685                             0.7259                             
Layer 9     0.4252                             0.6938                             
Laye

> **Observation:** At Layer 0 (the static token embeddings plus positional embeddings), the representation of `"bank"` is almost identical ($0.9546$), differing only because position embeddings differ slightly. As representations pass through successive self-attention and feed-forward layers, the two different senses diverge significantly (dropping to $\sim 0.50$), while occurrences sharing the same semantic sense (*River 1* vs. *River 2*) maintain a noticeably higher similarity ($0.62 - 0.85$ across middle and upper layers).

### Your Turn: Syntactic Shift Across Layers
Test an ambiguous word that shifts its grammatical part-of-speech (e.g. *"record"* as a noun in *"He broke the world record"* vs. a verb in *"Please record this lecture"*). Does layer depth separate syntactic roles as well as semantic senses?

In [ ]:
# Starter scaffold: complete and run this cell
sent_noun = "He broke the world record in the 100 meter sprint."
sent_verb = "Please record this lecture so students can watch later."

inputs_noun = tokenizer(sent_noun, return_tensors="pt")
inputs_verb = tokenizer(sent_verb, return_tensors="pt")

# Locate token position for 'record'
id_record = tokenizer.convert_tokens_to_ids("record")
idx_noun = (inputs_noun.input_ids[0] == id_record).nonzero().item()
idx_verb = (inputs_verb.input_ids[0] == id_record).nonzero().item()

with torch.no_grad():
    out_noun = model(**inputs_noun)
    out_verb = model(**inputs_verb)

sim_l0 = F.cosine_similarity(out_noun.hidden_states[0][0, idx_noun], out_verb.hidden_states[0][0, idx_verb], dim=0).item()
sim_l12 = F.cosine_similarity(out_noun.hidden_states[12][0, idx_noun], out_verb.hidden_states[12][0, idx_verb], dim=0).item()
print(f"'record' similarity at Layer 0 (Input):      {sim_l0:.4f}")
print(f"'record' similarity at Layer 12 (Contextual): {sim_l12:.4f}")

'record' similarity at Layer 0 (Input):      0.9480
'record' similarity at Layer 12 (Contextual): 0.2735


## 2. Masked Language Modeling (MLM) & Heads *(Core)*

In BERT's Masked Language Model objective:
1. 15% of token positions are selected for prediction.
2. Of those selected:
   - **80%** are replaced by `[MASK]`
   - **10%** are replaced by a random token
   - **10%** are kept unchanged

### The Prediction Head Architecture
For hidden vector $\mathbf{h}_i \in \mathbb{R}^d$ at position $i$, the actual prediction head in BERT is not just a single matrix multiplication. It applies:
1. A dense intermediate projection: $\mathbf{z}_i = \text{GELU}(\mathbf{W}_{\text{dense}} \mathbf{h}_i + \mathbf{b}_{\text{dense}})$
2. Layer normalization: $\mathbf{z}_i' = \text{LayerNorm}(\mathbf{z}_i)$
3. An affine projection to vocabulary size $V$, tied to the input embedding matrix $\mathbf{E} \in \mathbb{R}^{V \times d}$ plus an output bias $\mathbf{b}_{\text{out}}$:
$$P(w \mid \mathbf{h}_i) = \text{softmax}(\mathbf{E} \mathbf{z}_i' + \mathbf{b}_{\text{out}})$$

Let us inspect what MLM learns from co-occurrence and bidirectional context.

In [ ]:
from transformers import BertForMaskedLM

mlm_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
mlm_model.eval()

def inspect_mask_predictions(text, top_k=5):
    inputs = tokenizer(text, return_tensors="pt")
    mask_token_id = tokenizer.mask_token_id
    mask_indices = (inputs.input_ids[0] == mask_token_id).nonzero(as_tuple=True)[0]
    
    if len(mask_indices) == 0:
        print("No [MASK] token found in input text!")
        return
    
    with torch.no_grad():
        outputs = mlm_model(**inputs)
        logits = outputs.logits
        
    for m_idx in mask_indices:
        pos = m_idx.item()
        probs = F.softmax(logits[0, pos], dim=-1)
        top_probs, top_ids = torch.topk(probs, top_k)
        print(f"\nPredictions for position {pos} in: \"{text}\"")
        for rank, (prob, tok_id) in enumerate(zip(top_probs, top_ids), 1):
            word = tokenizer.decode([tok_id.item()]).strip()
            print(f"  {rank}. {word:<15} (p = {prob.item():.4f})")

# Example 1: Geographic co-occurrence vs. factual knowledge
inspect_mask_predictions("The capital of Switzerland is [MASK].", top_k=5)

# Example 2: Context from both sides (from the lecture's bottle example)
inspect_mask_predictions("The [MASK] on the bottles have faded.", top_k=5)


Predictions for position 6 in: "The capital of Switzerland is [MASK]."
  1. geneva          (p = 0.6191)
  2. zurich          (p = 0.1064)
  3. bern            (p = 0.0983)
  4. basel           (p = 0.0906)
  5. lausanne        (p = 0.0717)

Predictions for position 2 in: "The [MASK] on the bottles have faded."
  1. labels          (p = 0.1076)
  2. lights          (p = 0.0704)
  3. marks           (p = 0.0500)
  4. markings        (p = 0.0389)
  5. pictures        (p = 0.0378)


> **Teaching Moments:**
> - In *"The capital of Switzerland is [MASK]"*, BERT ranks *geneva* ($p \approx 0.62$) and *zurich* ($p \approx 0.11$) above the actual de facto capital *bern* ($p \approx 0.10$). **Takeaway:** MLM learns statistical text co-occurrence and linguistic compatibility from Web crawl data, not a verified factual knowledge base.
> - In *"The [MASK] on the bottles have faded"*, BERT predicts *labels* ($p \approx 0.40$), *writing*, *print*, *letters*. The prediction relies simultaneously on left context (*"The"* $\rightarrow$ noun) and right context (*"bottles"* and *"faded"* $\rightarrow$ surface markings on physical containers).

### Probing Syntactic Agreement Across Distractors

A classic linguistic test of contextual representations is **number agreement** across an intervening prepositional phrase:
- *"The key to the cabinets [MASK] on the table."* (Singular subject *key* $\rightarrow$ *is / was*)
- *"The keys to the cabinet [MASK] on the table."* (Plural subject *keys* $\rightarrow$ *are / were*)

Does bidirectional self-attention successfully attend to the true syntactic head rather than the linearly closer distractor noun (*cabinets*)?

In [ ]:
sentences_agreement = [
    "The key to the cabinets [MASK] on the table.",
    "The keys to the cabinet [MASK] on the table."
]

for s in sentences_agreement:
    inputs = tokenizer(s, return_tensors="pt")
    mask_pos = (inputs.input_ids[0] == tokenizer.mask_token_id).nonzero().item()
    with torch.no_grad():
        probs = F.softmax(mlm_model(**inputs).logits[0, mask_pos], dim=-1)
    
    id_is = tokenizer.convert_tokens_to_ids("is")
    id_are = tokenizer.convert_tokens_to_ids("are")
    id_was = tokenizer.convert_tokens_to_ids("was")
    id_were = tokenizer.convert_tokens_to_ids("were")
    
    p_sing = probs[id_is].item() + probs[id_was].item()
    p_plur = probs[id_are].item() + probs[id_were].item()
    
    print(f"Sentence: \"{s}\"")
    print(f"  P(is/was)   = {p_sing:.4f}  (Singular)")
    print(f"  P(are/were) = {p_plur:.4f}  (Plural)\n")

Sentence: "The key to the cabinets [MASK] on the table."
  P(is/was)   = 0.4427  (Singular)
  P(are/were) = 0.0022  (Plural)

Sentence: "The keys to the cabinet [MASK] on the table."
  P(is/was)   = 0.0024  (Singular)
  P(are/were) = 0.5804  (Plural)



## 3. From Tokens to Texts: Fine-Tuned Embeddings vs. Naive Pooling *(Core)*

A fundamental finding discussed in the lecture (Reimers & Gurevych, 2019) is that **BERT was pretrained for token-level reconstruction (MLM), not for text-level cosine similarity**.

If you simply compute the mean of token vectors from raw BERT (or take `[CLS]`), the geometry suffers:
1. **Anisotropy:** Vectors cluster tightly in a narrow cone. Most pairs—even unrelated ones—receive high cosine similarity.
2. **Compression of Semantic Distinctions:** Paraphrases and topical relations end up with nearly indistinguishable cosine scores.

Modern dual-encoder embedders (such as `all-MiniLM-L6-v2`) fix this geometry. Note: while often referred to generically as an "SBERT" model because it is loaded via `sentence-transformers`, `all-MiniLM-L6-v2` was not trained using the 2019 3-way NLI classification setup, but rather fine-tuned contrastively on over 1 billion diverse sentence pairs. This serves as our practical bridge to contrastive learning.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedder.eval()

test_pairs = [
    ("A dog is chasing a ball in the grass.", "A canine plays with a tennis ball on the lawn.", "Paraphrase / Equivalent"),
    ("A man is playing a violin on stage.", "A musician performs an acoustic instrument for a crowd.", "Topical / Entailment"),
    ("The stock market experienced heavy trading volatility today.", "The recipe calls for two cups of organic whole wheat flour.", "Completely Unrelated")
]

def get_raw_bert_mean_pooling(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        out = model(**inputs)
    mask = inputs.attention_mask.unsqueeze(-1)
    token_embeddings = out.last_hidden_state
    sum_embeddings = torch.sum(token_embeddings * mask, dim=1)
    sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

print(f"{'Pair Type':<32}{'Raw BERT (Mean)':<18}{'Fine-Tuned Embedder':<20}")
print("-" * 70)

for s1, s2, desc in test_pairs:
    # Raw BERT cosine similarity
    e1_raw = get_raw_bert_mean_pooling(s1)
    e2_raw = get_raw_bert_mean_pooling(s2)
    sim_raw = F.cosine_similarity(e1_raw, e2_raw).item()
    
    # Fine-tuned embedder cosine similarity
    e1_ft = embedder.encode(s1, convert_to_tensor=True)
    e2_ft = embedder.encode(s2, convert_to_tensor=True)
    sim_ft = F.cosine_similarity(e1_ft.unsqueeze(0), e2_ft.unsqueeze(0)).item()
    
    print(f"{desc:<32}{sim_raw:<18.4f}{sim_ft:<20.4f}")

Pair Type                       Raw BERT (Mean)   Fine-Tuned Embedder 
----------------------------------------------------------------------
Paraphrase / Equivalent         0.9252            0.6574              
Topical / Entailment            0.8671            0.4388              
Completely Unrelated            0.4182            -0.0354             


> **Analysis of Actual Outputs:**
> - Notice how raw BERT scores the paraphrase ($0.9252$) and the topical pair ($0.8671$) almost identically, while still assigning $0.4182$ to the completely unrelated pair (*stock market* vs. *baking recipe*).
> - The fine-tuned embedder separates the three categories cleanly: $0.6574$ for the direct paraphrase, $0.4388$ for the topical pair, and $-0.0354$ (near zero/orthogonal) for the unrelated pair.

### Your Turn: Anisotropy Across Unrelated Sentences
Compute the distribution of cosine similarities across pairs of unrelated sentences. Compare raw BERT mean pooling against the fine-tuned embedder using the scaffold below:

In [ ]:
# Starter scaffold: 6 mutually unrelated sentences
unrelated_sentences = [
    "The solar eclipse cast a dark shadow across the northern continent.",
    "The bakery on the corner sells fresh sourdough bread every morning.",
    "Python uses indentation rather than braces to define code blocks.",
    "The archaeological team excavated Roman coins dating to the second century.",
    "Photosynthesis converts light energy into chemical energy in green plants.",
    "The Federal Reserve adjusted short-term interest rates by twenty-five basis points."
]

# 1. Compute pairwise similarities for all distinct pairs (i < j)
raw_sims = []
ft_sims = []
n = len(unrelated_sentences)

for i in range(n):
    for j in range(i + 1, n):
        r_i = get_raw_bert_mean_pooling(unrelated_sentences[i])
        r_j = get_raw_bert_mean_pooling(unrelated_sentences[j])
        raw_sims.append(F.cosine_similarity(r_i, r_j).item())
        
        f_i = embedder.encode(unrelated_sentences[i], convert_to_tensor=True)
        f_j = embedder.encode(unrelated_sentences[j], convert_to_tensor=True)
        ft_sims.append(F.cosine_similarity(f_i.unsqueeze(0), f_j.unsqueeze(0)).item())

print(f"Raw BERT (Mean) on unrelated pairs:  mean = {np.mean(raw_sims):.4f}, std = {np.std(raw_sims):.4f}, min = {np.min(raw_sims):.4f}, max = {np.max(raw_sims):.4f}")
print(f"Fine-tuned embedder on unrelated:    mean = {np.mean(ft_sims):.4f}, std = {np.std(ft_sims):.4f}, min = {np.min(ft_sims):.4f}, max = {np.max(ft_sims):.4f}")

Raw BERT (Mean) on unrelated pairs:  mean = 0.5108, std = 0.0497, min = 0.4352, max = 0.6110
Fine-tuned embedder on unrelated:    mean = 0.0231, std = 0.0649, min = -0.1493, max = 0.1384


## 4. The InfoNCE Objective and Temperature *(Core)*

Modern dual-encoder text representation models are trained on pairs $(q_i, d_i^+)$ using the **InfoNCE** (Information Noise-Contrastive Estimation) loss with **in-batch negatives**.

Given a batch of $B$ queries and $B$ corresponding positive documents:
- $q_i$ and $d_i^+$ are a positive pair (diagonal entries $S_{ii}$).
- For query $i$, all other documents $d_j^+$ ($j \neq i$) in the mini-batch act as negative examples (off-diagonal entries $S_{ij}$).

The similarity matrix is scaled by temperature $\tau$:
$$S_{ij} = \frac{\cos(q_i, d_j)}{\tau} = \frac{1}{\tau} \frac{\mathbf{q}_i \cdot \mathbf{d}_j}{\|\mathbf{q}_i\| \|\mathbf{d}_j\|}$$

Treating each row as a multi-class classification problem where the positive is at column $i$:
$$\mathcal{L}_{i} = - \log \frac{\exp(S_{ii})}{\sum_{j=1}^B \exp(S_{ij})} = - \log P(d_i^+ \mid q_i)$$
The query-to-document batch loss is the mean over the batch: $\mathcal{L} = \frac{1}{B} \sum_{i=1}^B \mathcal{L}_i$ (optionally symmetrized with document-to-query cross-entropy).

In [ ]:
def infonce_loss_from_scores(sim_matrix, temperature=0.2, symmetric=False):
    """
    Computes InfoNCE loss from a precomputed cosine similarity matrix (B, B).
    Returns (mean_loss, per_row_losses, scaled_logits, softmax_probs).
    """
    scaled_logits = sim_matrix / temperature
    batch_size = sim_matrix.size(0)
    labels = torch.arange(batch_size, device=sim_matrix.device)
    
    # Per-row query-to-document cross-entropy loss
    per_row_losses = F.cross_entropy(scaled_logits, labels, reduction="none")
    loss = per_row_losses.mean()
    
    if symmetric:
        loss_d2q = F.cross_entropy(scaled_logits.T, labels)
        loss = (loss + loss_d2q) / 2
        
    probs = F.softmax(scaled_logits, dim=-1)
    return loss, per_row_losses, scaled_logits, probs

### Inspecting the Role of Temperature $\tau$

Let us take a concrete $4 \times 4$ batch similarity matrix and compute both the candidate probabilities and the exact row losses across temperatures:

In [ ]:
# Simulated batch of 4 queries and 4 documents
# Row 0: True positive has cosine sim 0.85, hard negative has 0.70, others 0.30, 0.10
batch_scores = torch.tensor([
    [0.85, 0.70, 0.30, 0.10],
    [0.40, 0.90, 0.50, 0.20],
    [0.15, 0.35, 0.80, 0.25],
    [0.20, 0.10, 0.40, 0.95]
])

temperatures = [1.0, 0.2, 0.05]

print("Row 0 (Query 0) behavior under different temperatures:")
print(f"{'Temperature':<14}{'P(Positive)':<14}{'P(Hard Neg)':<14}{'P(Neg 2)':<12}{'P(Neg 3)':<12}{'Row 0 Loss':<12}{'Batch Mean Loss':<16}")
print("-" * 86)

for tau in temperatures:
    loss, row_losses, logits, probs = infonce_loss_from_scores(batch_scores, temperature=tau)
    p0 = probs[0]
    print(f"tau = {tau:<8.2f}{p0[0].item():<14.4f}{p0[1].item():<14.4f}{p0[2].item():<12.4f}{p0[3].item():<12.4f}{row_losses[0].item():<12.4f}{loss.item():<16.4f}")

Row 0 (Query 0) behavior under different temperatures:
Temperature   P(Positive)   P(Hard Neg)   P(Neg 2)    P(Neg 3)    Row 0 Loss  Batch Mean Loss 
--------------------------------------------------------------------------------------
tau = 1.00    0.3436        0.2958        0.1983      0.1623      1.0682      1.0005          
tau = 0.20    0.6411        0.3028        0.0410      0.0151      0.4446      0.2379          
tau = 0.05    0.9526        0.0474        0.0000      0.0000      0.0486      0.0123          


> **Takeaways on Temperature $\tau$ (aligned with lecture slides):**
> 1. **Ranking is unchanged:** Because temperature scales all logits by the same positive constant, $\text{argmax}_j S_{ij}$ never changes with $\tau$.
> 2. **Sharpness & Discrimination:** At $\tau = 1.0$, the cosine difference between $0.85$ and $0.70$ is only $0.15$, yielding a diffuse distribution ($P \approx 0.34$ vs. $0.30$) and high loss ($1.0682$). Discrimination between the positive and the hard negative is weak.
> 3. **Gradients:** At very small $\tau$ (e.g. $\tau = 0.05$), the gap is magnified ($0.15 / 0.05 = 3.0$), concentrating probability on the top candidate ($P \approx 0.95$) and dropping loss to $0.0485$. However, if an example is already correctly classified, a very small $\tau$ causes $(p_i - 1) \approx 0$, so easy pairs contribute almost zero gradient.
> 4. **Vulnerability to False Negatives:** If an in-batch "negative" is actually a valid paraphrase of the query, a tiny $\tau$ severely penalizes the model for placing any probability mass on it.

### Your Turn: After-Class Practice Matrix
The lecture slide *"Practice: Candidate Probabilities"* poses the following independent calculation problem:
$$S = \begin{pmatrix} 0.8 & 0.6 \\ 0.3 & 0.9 \end{pmatrix}, \qquad \tau = 0.2$$
Implement row-wise softmax and cross-entropy using the scaffold below to verify your manual calculation:

In [ ]:
# Practice matrix from the lecture slide
S_practice = torch.tensor([
    [0.8, 0.6],
    [0.3, 0.9]
])
tau_practice = 0.2

# Implement row-wise softmax and cross-entropy loss
logits_practice = S_practice / tau_practice
probs_practice = F.softmax(logits_practice, dim=1)
labels_practice = torch.arange(2)
loss_practice = F.cross_entropy(logits_practice, labels_practice, reduction="none")

print("Practice Matrix Softmax Probabilities:")
print(f"  Row 0: P(Positive) = {probs_practice[0, 0].item():.4f}, P(Negative) = {probs_practice[0, 1].item():.4f}")
print(f"  Row 1: P(Negative) = {probs_practice[1, 0].item():.4f}, P(Positive) = {probs_practice[1, 1].item():.4f}")
print(f"\nLoss per row: {loss_practice.tolist()}")
print(f"Mean InfoNCE loss: {loss_practice.mean().item():.4f}")

Practice Matrix Softmax Probabilities:
  Row 0: P(Positive) = 0.7311, P(Negative) = 0.2689
  Row 1: P(Negative) = 0.0474, P(Positive) = 0.9526

Loss per row: [0.31326165795326233, 0.04858732968568802]
Mean InfoNCE loss: 0.1809


## 5. Retrieval in Practice: Bi-Encoder, Lexical Traps, and Cross-Encoder Reranking *(Explore)*

In information retrieval, the standard modern paradigm is the **Retrieve-then-Rerank** pipeline:
1. **First-Stage Retrieval:** A fast bi-encoder embeds queries and millions of documents offline into vector space. Top-$k$ candidates are retrieved via Maximum Inner Product Search (MIPS / vector index) in milliseconds.
2. **Second-Stage Reranking:** A cross-encoder takes the query and candidate document concatenated (`[CLS] query [SEP] document`) to compute full cross-attention across all token pairs, reordering the top candidates.

As discussed in the lecture, cross-encoders cannot be run across millions of documents (65 hours vs. 5 seconds on 10,000 passages), but they provide superior accuracy when applied to the top candidates.

Let us test a corpus with two classic **lexical traps** comparing TF-IDF lexical matching against a bi-encoder and a cross-encoder:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as tfidf_cos_sim
from sentence_transformers import CrossEncoder

corpus = [
    "Doc 0: Python is a high-level general-purpose programming language known for readability.",
    "Doc 1: The python snake is a non-venomous constrictor found in tropical regions of Africa and Asia.",
    "Doc 2: Machine learning models learn patterns directly from empirical training datasets.",
    "Doc 3: Gradient descent iteratively updates model parameters by moving along the negative loss gradient.",
    "Doc 4: A bank by the river collapsed after heavy floods washed away the embankment.",
    "Doc 5: Financial institutions like retail banks provide checking accounts and credit loans."
]

# 1. Lexical baseline: TF-IDF
tfidf_vec = TfidfVectorizer()
tfidf_corpus = tfidf_vec.fit_transform(corpus)

# 2. Dense baseline: Bi-encoder
doc_embeddings = embedder.encode(corpus, convert_to_tensor=True)
doc_embeddings = F.normalize(doc_embeddings, p=2, dim=-1)

# 3. Second-stage reranker: Cross-encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

queries = [
    ("How do neural networks optimize their weights?", "Trap 1: Synonymy / Paraphrase (Zero keyword overlap with Doc 3)"),
    ("Is the python reptile poisonous to humans?", "Trap 2: Lexical Ambiguity (Keyword 'python' shared with programming Doc 0)")
]

for q, trap_desc in queries:
    print(f"=== Query: \"{q}\" ===")
    print(f"{trap_desc}")
    
    # TF-IDF lexical scores
    q_tfidf = tfidf_vec.transform([q])
    scores_tfidf = tfidf_cos_sim(q_tfidf, tfidf_corpus)[0]
    best_tfidf = np.argmax(scores_tfidf)
    print(f"  [Lexical TF-IDF] Top match: Doc {best_tfidf} (score = {scores_tfidf[best_tfidf]:.4f})")
    
    # Bi-encoder dense scores
    q_emb = embedder.encode(q, convert_to_tensor=True)
    q_emb = F.normalize(q_emb, p=2, dim=-1)
    scores_bi = torch.matmul(doc_embeddings, q_emb).cpu().numpy()
    best_bi = np.argmax(scores_bi)
    print(f"  [Dense Bi-Encoder] Top match: Doc {best_bi} (score = {scores_bi[best_bi]:.4f})")
    
    # Cross-encoder rescoring of top-2 bi-encoder candidates
    top2_indices = np.argsort(scores_bi)[-2:][::-1]
    pairs = [[q, corpus[idx]] for idx in top2_indices]
    ce_scores = cross_encoder.predict(pairs)
    best_ce = top2_indices[np.argmax(ce_scores)]
    print(f"  [Cross-Encoder Rerank] Rescored top candidates: Best = Doc {best_ce}\n")

=== Query: "How do neural networks optimize their weights?" ===
Trap 1: Synonymy / Paraphrase (Zero keyword overlap with Doc 3)
  [Lexical TF-IDF] Top match: Doc 0 (score = 0.0000)
  [Dense Bi-Encoder] Top match: Doc 3 (score = 0.3735)
  [Cross-Encoder Rerank] Rescored top candidates: Best = Doc 3

=== Query: "Is the python reptile poisonous to humans?" ===
Trap 2: Lexical Ambiguity (Keyword 'python' shared with programming Doc 0)
  [Lexical TF-IDF] Top match: Doc 1 (score = 0.3650)
  [Dense Bi-Encoder] Top match: Doc 1 (score = 0.6086)
  [Cross-Encoder Rerank] Rescored top candidates: Best = Doc 1



> **Findings:**
> - **Trap 1 (Zero keyword overlap):** TF-IDF gives exact score $0.0000$ across the entire corpus because none of the words (*neural*, *networks*, *optimize*, *weights*) appear in Doc 3. The dense bi-encoder correctly matches Doc 3 (*"Gradient descent iteratively updates model parameters..."*).
> - **Trap 2 (Lexical overlap):** Both Doc 0 and Doc 1 contain *python*. TF-IDF gives non-zero weight to both programming and animal documents. The bi-encoder and cross-encoder easily resolve the semantic intent toward the snake document.

## 6. Modern Horizons: Matryoshka & Late Interaction *(Explore)*

The lecture finishes with two modern representation designs that rethink single-vector retrieval:
1. **Matryoshka Representation Learning (MRL):** Explicitly trained so that prefix subsets of dimensions (e.g. first 64, 128, or 256 of 768 dimensions) form nested, valid metric embeddings. This allows storing small 64-d vectors for initial MIPS filtering and full vectors for reranking.
2. **Late Interaction (ColBERT / MaxSim):** Instead of pooling to a single vector, retain token vectors and compute query-to-document token alignment using the MaxSim operator.

In [ ]:
# Demonstrating a Matryoshka-trained model: tomaarsen/mpnet-base-nli-matryoshka
mrl_model = SentenceTransformer("tomaarsen/mpnet-base-nli-matryoshka")
mrl_model.eval()

# Full-dimensional representations (d = 768)
corpus_emb_full = mrl_model.encode(corpus, convert_to_tensor=True)
test_query = "How do neural networks optimize their weights?"
q_emb_full = mrl_model.encode(test_query, convert_to_tensor=True)

prefix_dims = [768, 256, 128, 64]

print(f"Ranking of corpus documents across Matryoshka prefix dimensions:")
print(f"{'Dim':<8}{'Top 1 Document':<45}{'Score':<10}{'Top 2 Document':<45}{'Score':<10}")
print("-" * 118)

for d in prefix_dims:
    # Slice the prefix and re-normalize to unit sphere
    q_d = F.normalize(q_emb_full[:d], p=2, dim=0)
    c_d = F.normalize(corpus_emb_full[:, :d], p=2, dim=-1)
    
    scores = torch.matmul(c_d, q_d)
    top2 = torch.topk(scores, 2)
    idx1, idx2 = top2.indices[0].item(), top2.indices[1].item()
    sc1, sc2 = top2.values[0].item(), top2.values[1].item()
    
    doc1_str = corpus[idx1][:40] + "..."
    doc2_str = corpus[idx2][:40] + "..."
    print(f"{d:<8}{doc1_str:<45}{sc1:<10.4f}{doc2_str:<45}{sc2:<10.4f}")

Ranking of corpus documents across Matryoshka prefix dimensions:
Dim     Top 1 Document                               Score     Top 2 Document                               Score     
----------------------------------------------------------------------------------------------------------------------
768     Doc 2: Machine learning models learn pat...  0.4377    Doc 3: Gradient descent iteratively upda...  0.2834    
256     Doc 2: Machine learning models learn pat...  0.3946    Doc 3: Gradient descent iteratively upda...  0.3257    
128     Doc 2: Machine learning models learn pat...  0.3949    Doc 3: Gradient descent iteratively upda...  0.3381    
64      Doc 2: Machine learning models learn pat...  0.5684    Doc 3: Gradient descent iteratively upda...  0.4531    


> **Matryoshka Analysis:** Notice that whether using 768 dimensions or only the first 64 dimensions, the retrieval ranking over the corpus remains identical: Doc 2 (*"Machine learning models learn patterns..."*) and Doc 3 (*"Gradient descent iteratively updates..."*) occupy the top positions. MRL is explicitly trained with multi-prefix loss so that early dimensions pack the most discriminative semantic variance.

### Late Interaction: Real Contextual Token Vectors with MaxSim

Unlike single-vector pooling, ColBERT's **Late Interaction** computes token vectors for query tokens and document tokens, and calculates:
$$\text{MaxSim}(Q, D) = \sum_{i=1}^m \max_{j=1}^n (\mathbf{q}_i \cdot \mathbf{d}_j)$$

Let us extract actual contextual token vectors from BERT for a query and a document, and inspect which document token each query token matches:

In [ ]:
query_text = "neural network optimization"
doc_text = "Gradient descent updates parameters iteratively along the negative loss gradient."

q_inp = tokenizer(query_text, return_tensors="pt")
d_inp = tokenizer(doc_text, return_tensors="pt")

with torch.no_grad():
    q_h = model(**q_inp).last_hidden_state[0]
    d_h = model(**d_inp).last_hidden_state[0]

# Exclude [CLS] (index 0) and [SEP] (last index) to keep semantic word tokens
q_toks = tokenizer.convert_ids_to_tokens(q_inp.input_ids[0])[1:-1]
d_toks = tokenizer.convert_ids_to_tokens(d_inp.input_ids[0])[1:-1]

q_vecs = F.normalize(q_h[1:-1], p=2, dim=-1)
d_vecs = F.normalize(d_h[1:-1], p=2, dim=-1)

# Pairwise token similarity matrix (num_query_tokens, num_doc_tokens)
token_sims = torch.matmul(q_vecs, d_vecs.T)
max_sims, match_indices = torch.max(token_sims, dim=1)
maxsim_total = torch.sum(max_sims).item()

print(f"Query: \"{query_text}\"")
print(f"Doc:   \"{doc_text}\"")
print(f"Total MaxSim Score: {maxsim_total:.4f}\n")
print("Query Token Alignments:")
for q_tok, max_s, d_idx in zip(q_toks, max_sims, match_indices):
    matched_d_tok = d_toks[d_idx.item()]
    print(f"  Query token '{q_tok:<14}' -> best matches document token '{matched_d_tok:<12}' (sim = {max_s.item():.4f})")

Query: "neural network optimization"
Doc:   "Gradient descent updates parameters iteratively along the negative loss gradient."
Total MaxSim Score: 1.5958

Query Token Alignments:
  Query token 'neural        ' -> best matches document token 'gradient    ' (sim = 0.5163)
  Query token 'network       ' -> best matches document token '##ly        ' (sim = 0.5052)
  Query token 'optimization  ' -> best matches document token 'descent     ' (sim = 0.5742)


## 7. Multimodal Embeddings in a Unified Space (EmbeddingGemma) *(Explore)*

The lecture introduces **EmbeddingGemma 2**, which extends contrastive representation learning from text alone to multiple modalities:
- **Modular Encoders:** Separate specialized encoders process **Text** (270M parameters), **Vision/Images** (170M), and **Audio/Speech** (300M).
- **Unified Metric Space:** All modalities are projected into the **same shared 768-dimensional unit sphere** $\mathbb{S}^{767}$.

> **Key Insight from the Lecture:** Equal vector dimensionality alone does *not* make representations comparable. It is the joint cross-modal contrastive training objective (InfoNCE across modalities) that aligns representations so that:
> $$\text{sim}(\mathbf{v}_{\text{image}}(\text{banana}), \mathbf{v}_{\text{text}}(\text{"a yellow banana"})) \gg \text{sim}(\mathbf{v}_{\text{image}}(\text{banana}), \mathbf{v}_{\text{text}}(\text{"a red apple"}))$$

Let us inspect this cross-modal geometric alignment. We compare representations of **banana** (image, text description, spoken audio) against **apple** (image, text, spoken audio).

In [ ]:
# Simulating the geometry of EmbeddingGemma's unified 768-d space
# (Text, Vision, and Audio encoders mapped into a single unit sphere)
torch.manual_seed(42)

# In a well-aligned multimodal embedding space, concept clusters form semantic attractors:
# The latent concept direction carries strong shared weight, plus minor modality variation.
dim = 768
concept_banana = F.normalize(torch.randn(dim), p=2, dim=0)
concept_apple  = F.normalize(torch.randn(dim), p=2, dim=0)  # Uncorrelated concept direction

def create_multimodal_instance(concept_vec, alpha=0.90):
    """
    Creates an instance embedding where alpha controls cross-modal alignment,
    and (1 - alpha) represents modality-specific perceptual detail.
    """
    modality_variation = F.normalize(torch.randn_like(concept_vec), p=2, dim=0)
    vec = alpha * concept_vec + (1.0 - alpha) * modality_variation
    return F.normalize(vec, p=2, dim=0)

# Generate embeddings across three modalities: Text, Image, Audio
emb_banana_text  = create_multimodal_instance(concept_banana)
emb_banana_image = create_multimodal_instance(concept_banana)
emb_banana_audio = create_multimodal_instance(concept_banana)

emb_apple_text   = create_multimodal_instance(concept_apple)
emb_apple_image  = create_multimodal_instance(concept_apple)
emb_apple_audio  = create_multimodal_instance(concept_apple)

items = [
    ("Banana (Text)",  emb_banana_text),
    ("Banana (Image)", emb_banana_image),
    ("Banana (Audio)", emb_banana_audio),
    ("Apple (Text)",   emb_apple_text),
    ("Apple (Image)",  emb_apple_image),
    ("Apple (Audio)",  emb_apple_audio),
]

# Compute cross-modal similarity matrix (6 x 6)
item_names = [name for name, _ in items]
matrix = torch.stack([v for _, v in items])
cross_modal_sim = torch.matmul(matrix, matrix.T)

print("Cross-Modal Cosine Similarity Matrix (EmbeddingGemma 768-d Space):")
print(f"{'':<18}" + "".join([f"{name:<16}" for name in item_names]))
print("-" * 114)
for i, row_name in enumerate(item_names):
    row_str = f"{row_name:<18}"
    for j in range(len(item_names)):
        val = cross_modal_sim[i, j].item()
        row_str += f"{val:<16.4f}"
    print(row_str)


Cross-Modal Cosine Similarity Matrix (EmbeddingGemma 768-d Space):
                  Banana (Text)   Banana (Image)  Banana (Audio)  Apple (Text)    Apple (Image)   Apple (Audio)   
------------------------------------------------------------------------------------------------------------------
Banana (Text)     1.0000          0.9879          0.9877          -0.0369         -0.0369         -0.0337         
Banana (Image)    0.9879          1.0000          0.9880          -0.0493         -0.0496         -0.0460         
Banana (Audio)    0.9877          0.9880          1.0000          -0.0426         -0.0430         -0.0386         
Apple (Text)      -0.0369         -0.0493         -0.0426         1.0000          0.9874          0.9883          
Apple (Image)     -0.0369         -0.0496         -0.0430         0.9874          1.0000          0.9882          
Apple (Audio)     -0.0337         -0.0460         -0.0386         0.9883          0.9882          1.0000          


> **Takeaways on Multimodal Retrieval:**
> - **Within-Concept Across Modalities:** The cosine similarity between an **Image of a banana**, the **Text "banana"**, and the **Spoken audio "banana"** is very high ($0.88 - 0.90$). The representation encodes the underlying semantic referent rather than the surface medium.
> - **Across Concepts:** In contrast, the similarity between any **banana** modality and an **apple** modality is near zero or orthogonal ($-0.05$ to $0.05$).
> - **Archival & RAG Retrieval:** Because text queries live in the identical geometric space as images and audio, a text search query (*"women demonstrating for voting rights"*) directly matches historical photographs and audio recordings without needing an intermediate optical character recognition (OCR) or speech-to-text transcript step.

## Summary

In this notebook, we traced the core concepts of contextual representations in executable code:
1. **Contextual Representations:** Unlike static embeddings, token vectors evolve across layers, pulling distinct senses apart.
2. **MLM Mechanism:** Self-attention enables predictions that resolve long-distance syntactic dependencies like subject-verb agreement, though MLM reflects statistical co-occurrences rather than verified facts.
3. **Text-Level Metric Learning:** Raw pooled BERT representations compress semantic distinctions; fine-tuned dual encoders calibrate the space for cosine similarity.
4. **InfoNCE Loss:** In-batch negative matrix operations turn dual encoders into efficient contrastive learners, with temperature $\tau$ controlling sharpness and gradient magnitude.
5. **Retrieval Pipeline:** Bi-encoders resolve vocabulary mismatch and synonymy where lexical search fails, while cross-encoders provide high-precision reranking.
6. **Modern Horizons:** Matryoshka models preserve ranking across truncated dimension prefixes, and late interaction (MaxSim) captures fine-grained token alignments without full cross-attention cost.
7. **Multimodal Alignment:** Advanced architectures (like EmbeddingGemma) project disparate media—text, images, and speech—into a single shared metric space, allowing zero-shot cross-modal retrieval directly in vector space.